# PySpark on Network/IP Traffic Data

Count, aggregate, join, sort, and search — applied directly to synthetic IP/cell-tower traffic logs, then used to flag suspicious IPs (fraud/anomaly candidates).

## Setup

In [ ]:
!pip -q install pyspark

In [ ]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
sc = SparkContext(appName="ip_traffic")
spark = SparkSession.Builder().getOrCreate()

## Generate the Traffic Data

Standing in for real logs: `(ip_address, tower_id, mb_sent)` rows. Most IPs behave normally, two (`10.0.0.99`, `10.0.0.100`) are made to send abnormally high volumes on purpose, so there's something real for the detection steps later on to actually catch.

In [ ]:
import random
random.seed(42)

towers = ["TWR-001", "TWR-002", "TWR-003", "TWR-004"]
normal_ips = [f"10.0.0.{i}" for i in range(1, 30)]
suspicious_ips = ["10.0.0.99", "10.0.0.100"]

logs = []
for ip in normal_ips:
    tower = random.choice(towers)
    for _ in range(random.randint(3, 8)):
        logs.append((ip, tower, random.randint(1, 20)))  # normal usage: 1-20 MB per connection

for ip in suspicious_ips:
    tower = random.choice(towers)
    for _ in range(random.randint(20, 40)):
        logs.append((ip, tower, random.randint(150, 400)))  # abnormally high usage

logs_df = spark.createDataFrame(logs, ["ip_address", "tower_id", "mb_sent"])
logs_df.show(5)

+----------+--------+-------+
|ip_address|tower_id|mb_sent|
+----------+--------+-------+
|  10.0.0.1| TWR-001|      9|
|  10.0.0.1| TWR-001|      8|
|  10.0.0.1| TWR-001|      8|
|  10.0.0.2| TWR-002|      4|
|  10.0.0.2| TWR-002|     18|
+----------+--------+-------+
only showing top 5 rows


# Part 1: Counting IPs

The core word-count pattern — `map` to `(key, 1)`, `reduceByKey` to sum, `sortBy` to rank — run on IP addresses instead of words. How often an IP shows up (its connection count) is a signal on its own, separate from how much data it sent.

### Basic count

In [ ]:
ip_rdd = sc.parallelize(logs).map(lambda row: row[0])

ip_counts = ip_rdd.map(lambda ip: (ip, 1)).reduceByKey(lambda a, b: a + b)

sorted_ip_counts = ip_counts.sortBy(lambda ic: ic[1], ascending=False)

print("Top 15 IPs by number of connections:")
for ip, n in sorted_ip_counts.take(15):
    print(ip, ":", n)

Top 15 IPs by number of connections:
10.0.0.99 : 33
10.0.0.100 : 21
10.0.0.8 : 8
10.0.0.17 : 8
10.0.0.2 : 8
10.0.0.4 : 8
10.0.0.11 : 8
10.0.0.19 : 8
10.0.0.7 : 7
10.0.0.12 : 7
10.0.0.14 : 7
10.0.0.20 : 7
10.0.0.22 : 7
10.0.0.10 : 7
10.0.0.9 : 6


### Variation 1: IPs connected to a specific tower

Filter the raw logs down to one tower, then count IPs on it — same shape as filtering words starting with "h" or "m".

In [ ]:
def ips_on_tower(tower_id):
    return (
        sc.parallelize(logs)
          .filter(lambda row: row[1] == tower_id)
          .map(lambda row: (row[0], 1))
          .reduceByKey(lambda a, b: a + b)
          .sortBy(lambda ic: ic[1], ascending=False)
    )

print("IPs on TWR-001:")
for ip, n in ips_on_tower("TWR-001").collect():
    print(ip, ":", n)

IPs on TWR-001:
10.0.0.99 : 33
10.0.0.100 : 21
10.0.0.17 : 8
10.0.0.19 : 8
10.0.0.20 : 7
10.0.0.22 : 7
10.0.0.10 : 7
10.0.0.6 : 6
10.0.0.16 : 6
10.0.0.21 : 4
10.0.0.27 : 3
10.0.0.1 : 3
10.0.0.15 : 3


### Variation 2: IPs with more than N connections

Same shape as filtering three/four-letter words — here, filtering on connection count instead of word length.

In [ ]:
MIN_CONNECTIONS = 15

heavy_connectors = ip_counts.filter(lambda ic: ic[1] > MIN_CONNECTIONS)

print(f"IPs with more than {MIN_CONNECTIONS} connections:")
for ip, n in heavy_connectors.sortBy(lambda ic: ic[1], ascending=False).collect():
    print(ip, ":", n)

IPs with more than 15 connections:
10.0.0.99 : 33
10.0.0.100 : 21


### Search: how many connections did a specific IP make?

In [ ]:
def search_ip(ip, ip_counts_rdd):
    result = ip_counts_rdd.filter(lambda ic: ic[0] == ip).collect()
    return result[0][1] if result else 0

for ip in ["10.0.0.99", "10.0.0.100", "10.0.0.1", "10.0.0.5"]:
    print(ip, ":", search_ip(ip, ip_counts))

10.0.0.99 : 33
10.0.0.100 : 21
10.0.0.1 : 3
10.0.0.5 : 5


# Part 2: Aggregates, Joins, Sorting, Searching (DataFrame API)

Same idea as Part 1, but this time aggregating **data volume** (`mb_sent`) instead of connection count, using the DataFrame API — `groupBy/agg`, `join`, `orderBy`, `filter`.

### Aggregate — total data per IP

Grouped on `ip_address` alone; each IP's tower is carried along with `F.first()` purely so the next step can look up that tower's baseline.

In [ ]:
ip_totals = (
    logs_df.groupBy("ip_address")
           .agg(
               F.first("tower_id").alias("tower_id"),
               F.sum("mb_sent").alias("total_mb"),
               F.count("*").alias("num_connections"),
           )
)
ip_totals.orderBy(F.desc("total_mb")).show(10)

+----------+--------+--------+---------------+
|ip_address|tower_id|total_mb|num_connections|
+----------+--------+--------+---------------+
| 10.0.0.99| TWR-001|    8605|             33|
|10.0.0.100| TWR-001|    5803|             21|
| 10.0.0.19| TWR-001|      95|              8|
| 10.0.0.16| TWR-001|      92|              6|
|  10.0.0.4| TWR-002|      90|              8|
|  10.0.0.7| TWR-004|      80|              7|
| 10.0.0.12| TWR-002|      76|              7|
| 10.0.0.17| TWR-001|      76|              8|
| 10.0.0.22| TWR-001|      76|              7|
| 10.0.0.10| TWR-001|      75|              7|
+----------+--------+--------+---------------+
only showing top 10 rows


### Join — attach each tower's expected baseline

In [ ]:
tower_baseline = spark.createDataFrame(
    [("TWR-001", 80), ("TWR-002", 80), ("TWR-003", 80), ("TWR-004", 80)],
    ["tower_id", "expected_max_mb"],
)

joined = ip_totals.join(tower_baseline, on="tower_id", how="left")
joined = joined.withColumn("ratio_to_baseline", F.round(F.col("total_mb") / F.col("expected_max_mb"), 2))
joined.orderBy(F.desc("ratio_to_baseline")).show(10)

+--------+----------+--------+---------------+---------------+-----------------+
|tower_id|ip_address|total_mb|num_connections|expected_max_mb|ratio_to_baseline|
+--------+----------+--------+---------------+---------------+-----------------+
| TWR-001| 10.0.0.99|    8605|             33|             80|           107.56|
| TWR-001|10.0.0.100|    5803|             21|             80|            72.54|
| TWR-001| 10.0.0.19|      95|              8|             80|             1.19|
| TWR-001| 10.0.0.16|      92|              6|             80|             1.15|
| TWR-002|  10.0.0.4|      90|              8|             80|             1.13|
| TWR-004|  10.0.0.7|      80|              7|             80|              1.0|
| TWR-002| 10.0.0.12|      76|              7|             80|             0.95|
| TWR-001| 10.0.0.17|      76|              8|             80|             0.95|
| TWR-001| 10.0.0.22|      76|              7|             80|             0.95|
| TWR-001| 10.0.0.10|      7

### Sort

In [ ]:
joined.orderBy(F.desc("total_mb")).show(10)

+--------+----------+--------+---------------+---------------+-----------------+
|tower_id|ip_address|total_mb|num_connections|expected_max_mb|ratio_to_baseline|
+--------+----------+--------+---------------+---------------+-----------------+
| TWR-001| 10.0.0.99|    8605|             33|             80|           107.56|
| TWR-001|10.0.0.100|    5803|             21|             80|            72.54|
| TWR-001| 10.0.0.19|      95|              8|             80|             1.19|
| TWR-001| 10.0.0.16|      92|              6|             80|             1.15|
| TWR-002|  10.0.0.4|      90|              8|             80|             1.13|
| TWR-004|  10.0.0.7|      80|              7|             80|              1.0|
| TWR-002| 10.0.0.12|      76|              7|             80|             0.95|
| TWR-001| 10.0.0.17|      76|              8|             80|             0.95|
| TWR-001| 10.0.0.22|      76|              7|             80|             0.95|
| TWR-001| 10.0.0.10|      7

### Search

In [ ]:
# exact match
joined.filter(F.col("ip_address") == "10.0.0.99").show()

# range: moderately high usage, not yet extreme
joined.filter((F.col("total_mb") >= 50) & (F.col("total_mb") <= 150)).show()

# pattern: ips in a particular /24-style prefix
joined.filter(F.col("ip_address").like("10.0.0.9%")).orderBy(F.desc("total_mb")).show()

+--------+----------+--------+---------------+---------------+-----------------+
|tower_id|ip_address|total_mb|num_connections|expected_max_mb|ratio_to_baseline|
+--------+----------+--------+---------------+---------------+-----------------+
| TWR-001| 10.0.0.99|    8605|             33|             80|           107.56|
+--------+----------+--------+---------------+---------------+-----------------+

+--------+----------+--------+---------------+---------------+-----------------+
|tower_id|ip_address|total_mb|num_connections|expected_max_mb|ratio_to_baseline|
+--------+----------+--------+---------------+---------------+-----------------+
| TWR-001| 10.0.0.10|      75|              7|             80|             0.94|
| TWR-002| 10.0.0.11|      59|              8|             80|             0.74|
| TWR-002| 10.0.0.12|      76|              7|             80|             0.95|
| TWR-002| 10.0.0.13|      64|              4|             80|              0.8|
| TWR-004| 10.0.0.14|      

### Flag and threshold

In [ ]:
FLAG_THRESHOLD = 3.0

flagged = joined.filter(F.col("ratio_to_baseline") > FLAG_THRESHOLD).orderBy(F.desc("ratio_to_baseline"))

print(f"Flagged {flagged.count()} ips (>{FLAG_THRESHOLD}x baseline):")
flagged.show()

Flagged 2 ips (>3.0x baseline):
+--------+----------+--------+---------------+---------------+-----------------+
|tower_id|ip_address|total_mb|num_connections|expected_max_mb|ratio_to_baseline|
+--------+----------+--------+---------------+---------------+-----------------+
| TWR-001| 10.0.0.99|    8605|             33|             80|           107.56|
| TWR-001|10.0.0.100|    5803|             21|             80|            72.54|
+--------+----------+--------+---------------+---------------+-----------------+



### Highlight flagged IPs in red

In [ ]:
import pandas as pd

def show_in_red(spark_df, caption=""):
    pdf = spark_df.toPandas()
    styled = pdf.style.set_properties(**{
        "background-color": "#ffe0e0",
        "color": "#b30000",
        "font-weight": "bold",
    })
    if caption:
        styled = styled.set_caption(caption)
    return styled

show_in_red(flagged, caption="Flagged IPs (simple threshold)")

,tower_id,ip_address,total_mb,num_connections,expected_max_mb,ratio_to_baseline
0,TWR-001,10.0.0.99,8605,33,80,107.560000
1,TWR-001,10.0.0.100,5803,21,80,72.540000


# Part 3: The Same Thing Using the RDD API

Same aggregate/join/sort/search, keyed on `ip_address` throughout, built with `map`/`reduceByKey`/`join`/`sortBy`/`filter` instead of DataFrame calls — matching the style used in Part 1.

### Aggregate — total MB per IP

In [ ]:
logs_rdd = sc.parallelize(logs)

pair_rdd = logs_rdd.map(lambda row: (row[0], row[2]))
ip_totals_rdd = pair_rdd.reduceByKey(lambda a, b: a + b)

# each ip only ever connects through one tower in this data, so this recovers it for the join below
ip_tower_rdd = logs_rdd.map(lambda row: (row[0], row[1])).distinct()

ip_totals_rdd.take(5)

[('10.0.0.7', 80),
 ('10.0.0.8', 51),
 ('10.0.0.9', 58),
 ('10.0.0.12', 76),
 ('10.0.0.13', 64)]

### Join — attach tower, then baseline

Two joins: #1 keyed on `ip_address` (attach each ip's tower), #2 keyed on `tower_id` (attach that tower's baseline — unavoidable since the baseline belongs to the tower, not the ip).

In [ ]:
tower_baseline_rdd = sc.parallelize([
    ("TWR-001", 80), ("TWR-002", 80), ("TWR-003", 80), ("TWR-004", 80),
])

# join #1, keyed on ip_address -> (ip, (total_mb, tower_id))
with_tower_rdd = ip_totals_rdd.join(ip_tower_rdd)

# re-key by tower_id to join the baseline
by_tower_rdd = with_tower_rdd.map(lambda kv: (kv[1][1], (kv[0], kv[1][0])))

# join #2, keyed on tower_id -> (tower_id, ((ip, total_mb), expected_mb))
joined_rdd = by_tower_rdd.join(tower_baseline_rdd)

scored_rdd = joined_rdd.map(lambda kv: {
    "tower": kv[0],
    "ip": kv[1][0][0],
    "total_mb": kv[1][0][1],
    "expected_mb": kv[1][1],
    "ratio": round(kv[1][0][1] / kv[1][1], 2),
})
scored_rdd.take(5)

[{'tower': 'TWR-003',
  'ip': '10.0.0.24',
  'total_mb': 55,
  'expected_mb': 80,
  'ratio': 0.69},
 {'tower': 'TWR-003',
  'ip': '10.0.0.25',
  'total_mb': 45,
  'expected_mb': 80,
  'ratio': 0.56},
 {'tower': 'TWR-003',
  'ip': '10.0.0.9',
  'total_mb': 58,
  'expected_mb': 80,
  'ratio': 0.72},
 {'tower': 'TWR-002',
  'ip': '10.0.0.8',
  'total_mb': 51,
  'expected_mb': 80,
  'ratio': 0.64},
 {'tower': 'TWR-002',
  'ip': '10.0.0.26',
  'total_mb': 13,
  'expected_mb': 80,
  'ratio': 0.16}]

### Sort

In [ ]:
sorted_rdd = scored_rdd.sortBy(lambda r: r["ratio"], ascending=False)

for r in sorted_rdd.take(10):
    print(r)

{'tower': 'TWR-001', 'ip': '10.0.0.99', 'total_mb': 8605, 'expected_mb': 80, 'ratio': 107.56}
{'tower': 'TWR-001', 'ip': '10.0.0.100', 'total_mb': 5803, 'expected_mb': 80, 'ratio': 72.54}
{'tower': 'TWR-001', 'ip': '10.0.0.19', 'total_mb': 95, 'expected_mb': 80, 'ratio': 1.19}
{'tower': 'TWR-001', 'ip': '10.0.0.16', 'total_mb': 92, 'expected_mb': 80, 'ratio': 1.15}
{'tower': 'TWR-002', 'ip': '10.0.0.4', 'total_mb': 90, 'expected_mb': 80, 'ratio': 1.12}
{'tower': 'TWR-004', 'ip': '10.0.0.7', 'total_mb': 80, 'expected_mb': 80, 'ratio': 1.0}
{'tower': 'TWR-002', 'ip': '10.0.0.12', 'total_mb': 76, 'expected_mb': 80, 'ratio': 0.95}
{'tower': 'TWR-001', 'ip': '10.0.0.17', 'total_mb': 76, 'expected_mb': 80, 'ratio': 0.95}
{'tower': 'TWR-001', 'ip': '10.0.0.22', 'total_mb': 76, 'expected_mb': 80, 'ratio': 0.95}
{'tower': 'TWR-001', 'ip': '10.0.0.10', 'total_mb': 75, 'expected_mb': 80, 'ratio': 0.94}


### Search

In [ ]:
FLAG_THRESHOLD = 3.0

flagged_rdd = scored_rdd.filter(lambda r: r["ratio"] > FLAG_THRESHOLD)

print(f"Flagged {flagged_rdd.count()} ips (>{FLAG_THRESHOLD}x baseline):")
for r in flagged_rdd.collect():
    print(r)

Flagged 2 ips (>3.0x baseline):
{'tower': 'TWR-001', 'ip': '10.0.0.99', 'total_mb': 8605, 'expected_mb': 80, 'ratio': 107.56}
{'tower': 'TWR-001', 'ip': '10.0.0.100', 'total_mb': 5803, 'expected_mb': 80, 'ratio': 72.54}


# Part 4: Realistic Time-Windowed Data + Two Fraud Detectors

A full week of hourly traffic instead of a one-shot batch: a daily busy/quiet curve, ordinary users, a few heavy-but-legitimate users (to test for false positives), IPs that suddenly spike and stay high, and IPs that ramp up slowly across the week (the pattern a naive threshold tends to miss).

In [ ]:
import math
from datetime import datetime, timedelta

random.seed(7)

START = datetime(2026, 8, 17, 0, 0, 0)
DAYS = 7
TOWERS = {"TWR-001": 1.0, "TWR-002": 1.0, "TWR-003": 1.3, "TWR-004": 0.8}

HOUR_WEIGHTS = [0.20, 0.15, 0.10, 0.10, 0.10, 0.15, 0.30, 0.50, 0.70, 0.85, 0.90, 0.95,
                1.00, 0.95, 0.90, 0.85, 0.90, 0.95, 1.00, 0.90, 0.70, 0.50, 0.35, 0.25]

def gen_ip_pool(n, prefix):
    return [f"10.0.{prefix}.{i}" for i in range(1, n + 1)]

normal_ips      = gen_ip_pool(40, 0)
heavy_legit_ips = gen_ip_pool(3, 1)
fraud_spike_ips = gen_ip_pool(2, 2)
fraud_ramp_ips  = gen_ip_pool(2, 3)

all_ips = normal_ips + heavy_legit_ips + fraud_spike_ips + fraud_ramp_ips
ip_tower = {ip: random.choice(list(TOWERS)) for ip in all_ips}

week_logs = []

for day in range(DAYS):
    for hour in range(24):
        ts = START + timedelta(days=day, hours=hour)
        w = HOUR_WEIGHTS[hour]

        for ip in normal_ips:
            n_conn = max(0, round(random.gauss(w * 2.5, 1)))
            for _ in range(n_conn):
                mb = random.lognormvariate(math.log(6), 0.5)
                week_logs.append((ip, ip_tower[ip], ts, round(mb, 2)))

        for ip in heavy_legit_ips:
            n_conn = max(0, round(random.gauss(w * 4, 1.5)))
            for _ in range(n_conn):
                mb = random.lognormvariate(math.log(35), 0.4)
                week_logs.append((ip, ip_tower[ip], ts, round(mb, 2)))

        for ip in fraud_spike_ips:
            if day < 5:
                n_conn = max(0, round(random.gauss(w * 2.5, 1)))
                mb_mu = math.log(6)
            else:
                n_conn = random.randint(15, 25)
                mb_mu = math.log(180)
            for _ in range(n_conn):
                mb = random.lognormvariate(mb_mu, 0.3)
                week_logs.append((ip, ip_tower[ip], ts, round(mb, 2)))

        for ip in fraud_ramp_ips:
            progress = (day * 24 + hour) / (DAYS * 24)
            n_conn = max(0, round(random.gauss(w * (2.5 + 12 * progress), 1)))
            mb_mu = math.log(6 + 90 * progress)
            for _ in range(n_conn):
                mb = random.lognormvariate(mb_mu, 0.35)
                week_logs.append((ip, ip_tower[ip], ts, round(mb, 2)))

print("total log rows:", len(week_logs))
week_logs_df = spark.createDataFrame(week_logs, ["ip_address", "tower_id", "timestamp", "mb_sent"])
week_logs_df.show(5)

total log rows: 15747
+----------+--------+-------------------+-------+
|ip_address|tower_id|          timestamp|mb_sent|
+----------+--------+-------------------+-------+
|  10.0.0.1| TWR-003|2026-08-17 00:00:00|   3.58|
|  10.0.0.7| TWR-003|2026-08-17 00:00:00|   4.49|
|  10.0.0.9| TWR-002|2026-08-17 00:00:00|  10.21|
| 10.0.0.10| TWR-001|2026-08-17 00:00:00|   9.59|
| 10.0.0.15| TWR-002|2026-08-17 00:00:00|   5.62|
+----------+--------+-------------------+-------+
only showing top 5 rows


## Step 1 — Aggregate into hourly windows, keyed on IP address

In [ ]:
hourly = (
    week_logs_df.withColumn("hour_bucket", F.date_trunc("hour", "timestamp"))
                .withColumn("hour_of_day", F.hour("timestamp"))
                .groupBy("ip_address", "hour_bucket", "hour_of_day")
                .agg(
                    F.first("tower_id").alias("tower_id"),
                    F.sum("mb_sent").alias("total_mb"),
                    F.count("*").alias("num_connections"),
                )
)
hourly.orderBy("ip_address", "hour_bucket").show(10)

+----------+-------------------+-----------+--------+--------+---------------+
|ip_address|        hour_bucket|hour_of_day|tower_id|total_mb|num_connections|
+----------+-------------------+-----------+--------+--------+---------------+
|  10.0.0.1|2026-08-17 00:00:00|          0| TWR-003|    3.58|              1|
|  10.0.0.1|2026-08-17 01:00:00|          1| TWR-003|    6.48|              2|
|  10.0.0.1|2026-08-17 02:00:00|          2| TWR-003|    2.11|              1|
|  10.0.0.1|2026-08-17 04:00:00|          4| TWR-003|    5.65|              1|
|  10.0.0.1|2026-08-17 05:00:00|          5| TWR-003|   11.24|              1|
|  10.0.0.1|2026-08-17 06:00:00|          6| TWR-003|    7.04|              1|
|  10.0.0.1|2026-08-17 08:00:00|          8| TWR-003|    4.74|              1|
|  10.0.0.1|2026-08-17 09:00:00|          9| TWR-003|   17.96|              3|
|  10.0.0.1|2026-08-17 10:00:00|         10| TWR-003|   17.69|              3|
|  10.0.0.1|2026-08-17 11:00:00|         11| TWR-003

## Step 2 — Diurnal-aware baseline

Per tower, per hour-of-day, not one flat number per tower. **Caveat**: computed from data that includes the fraud/heavy IPs, so it's pulled upward by them — a real pipeline would use a trusted historical period or robust stats (median/MAD) instead.

In [ ]:
baseline = (
    hourly.groupBy("tower_id", "hour_of_day")
          .agg(
              F.avg("total_mb").alias("avg_mb"),
              F.stddev("total_mb").alias("stddev_mb"),
          )
)
baseline = baseline.fillna({"stddev_mb": 1.0})
baseline.orderBy("tower_id", "hour_of_day").show(10)

+--------+-----------+------------------+------------------+
|tower_id|hour_of_day|            avg_mb|         stddev_mb|
+--------+-----------+------------------+------------------+
| TWR-001|          0|  9.87949152542373| 6.074926826217125|
| TWR-001|          1| 9.070317460317456| 5.250963957050039|
| TWR-001|          2| 8.670196078431372|5.3755846157217455|
| TWR-001|          3| 8.901521739130436|6.0201993202112885|
| TWR-001|          4|  9.54688524590164| 5.488185049414061|
| TWR-001|          5|  8.76593220338983| 6.256214047027237|
| TWR-001|          6| 9.656986301369862| 5.412384636936207|
| TWR-001|          7|12.694711538461549|  7.37633238999115|
| TWR-001|          8|14.363243243243236| 8.872755148836461|
| TWR-001|          9| 14.99974789915966| 8.220591184789084|
+--------+-----------+------------------+------------------+
only showing top 10 rows


## Step 3 — Join and score each hour with a z-score

In [ ]:
scored = (
    hourly.join(baseline, on=["tower_id", "hour_of_day"], how="left")
          .withColumn("z_score", F.round((F.col("total_mb") - F.col("avg_mb")) / F.col("stddev_mb"), 2))
)
scored.orderBy(F.desc("z_score")).show(10)

+--------+-----------+----------+-------------------+------------------+---------------+------------------+------------------+-------+
|tower_id|hour_of_day|ip_address|        hour_bucket|          total_mb|num_connections|            avg_mb|         stddev_mb|z_score|
+--------+-----------+----------+-------------------+------------------+---------------+------------------+------------------+-------+
| TWR-004|         11|  10.0.2.1|2026-08-22 11:00:00| 4615.990000000001|             24|155.99188405797102| 658.7328303048196|   6.77|
| TWR-004|         10|  10.0.2.1|2026-08-22 10:00:00|4100.4800000000005|             23| 144.7373134328358|  597.340568001665|   6.62|
| TWR-002|         11|  10.0.3.2|2026-08-23 11:00:00|1543.6999999999998|             15| 71.68027777777777|224.58316110213153|   6.55|
| TWR-002|         15|  10.0.3.2|2026-08-23 15:00:00|1404.2900000000002|             12| 65.88082191780822| 205.2918522274611|   6.52|
| TWR-002|         10|  10.0.3.2|2026-08-23 10:00:00|11

## Step 4 — Detector 1: sustained hourly spikes

Require several **consecutive** anomalous hours rather than flagging any single noisy hour.

In [ ]:
from pyspark.sql.window import Window

Z_THRESHOLD = 3.0
MIN_CONSECUTIVE_HOURS = 3

anomalous_hours = scored.filter(F.col("z_score") > Z_THRESHOLD)

w = Window.partitionBy("ip_address").orderBy("hour_bucket")
anomalous_hours = anomalous_hours.withColumn("rn", F.row_number().over(w))
anomalous_hours = anomalous_hours.withColumn(
    "run_id", F.col("hour_bucket").cast("long") - F.col("rn") * 3600
)

flagged_spikes = (
    anomalous_hours.groupBy("ip_address", "run_id")
                   .agg(
                       F.count("*").alias("consecutive_anomalous_hours"),
                       F.min("hour_bucket").alias("run_start"),
                       F.max("hour_bucket").alias("run_end"),
                       F.max("z_score").alias("peak_z_score"),
                   )
    .filter(F.col("consecutive_anomalous_hours") >= MIN_CONSECUTIVE_HOURS)
    .orderBy(F.desc("consecutive_anomalous_hours"))
)

show_in_red(flagged_spikes, caption="Flagged for sustained spikes")

,ip_address,run_id,consecutive_anomalous_hours,run_start,run_end,peak_z_score
0,10.0.2.1,1787353200,24,2026-08-22 00:00:00,2026-08-22 23:00:00,6.770000
1,10.0.3.2,1787349600,24,2026-08-23 00:00:00,2026-08-23 23:00:00,6.550000
2,10.0.2.1,1787356800,23,2026-08-23 01:00:00,2026-08-23 23:00:00,6.450000
3,10.0.2.2,1787360400,21,2026-08-22 05:00:00,2026-08-23 01:00:00,5.360000
4,10.0.2.2,1787367600,18,2026-08-23 06:00:00,2026-08-23 23:00:00,5.910000
5,10.0.3.2,1787342400,16,2026-08-22 06:00:00,2026-08-22 21:00:00,5.940000
6,10.0.2.2,1787353200,3,2026-08-22 00:00:00,2026-08-22 02:00:00,4.080000
7,10.0.3.2,1787295600,3,2026-08-21 08:00:00,2026-08-21 10:00:00,3.890000


## Step 5 — Detector 2: gradual ramps

Point-in-time z-scores tend to miss a slow ramp, since no single hour looks extreme against a baseline that's also slowly rising. Instead, track each IP's **day-over-day growth** with `lag()`.

In [ ]:
daily = (
    week_logs_df.withColumn("day", F.to_date("timestamp"))
                .groupBy("ip_address", "day")
                .agg(F.sum("mb_sent").alias("daily_mb"))
)

w2 = Window.partitionBy("ip_address").orderBy("day")
daily = daily.withColumn("prev_day_mb", F.lag("daily_mb").over(w2))
daily = daily.withColumn("growth_ratio", F.round(F.col("daily_mb") / F.col("prev_day_mb"), 2))

GROWTH_THRESHOLD = 1.15
MIN_GROWTH_DAYS = 4

growth_days = daily.filter(F.col("growth_ratio") >= GROWTH_THRESHOLD)

w3 = Window.partitionBy("ip_address").orderBy("day")
growth_days = growth_days.withColumn("rn", F.row_number().over(w3))
growth_days = growth_days.withColumn(
    "run_id", F.datediff(F.col("day"), F.lit(START.date())) - F.col("rn")
)

flagged_ramps = (
    growth_days.groupBy("ip_address", "run_id")
               .agg(
                   F.count("*").alias("consecutive_growth_days"),
                   F.min("day").alias("run_start"),
                   F.max("day").alias("run_end"),
               )
    .filter(F.col("consecutive_growth_days") >= MIN_GROWTH_DAYS)
    .orderBy(F.desc("consecutive_growth_days"))
)

show_in_red(flagged_ramps, caption="Flagged for sustained day-over-day growth (ramp pattern)")

,ip_address,run_id,consecutive_growth_days,run_start,run_end
0,10.0.3.1,0,6,2026-08-18,2026-08-23
1,10.0.3.2,0,6,2026-08-18,2026-08-23


## Notes

- **Two fraud shapes need two detectors.** Sustained z-scores catch sudden spikes; day-over-day growth catches gradual ramps. Neither alone covers both.
- **Baseline contamination is real.** These baselines are computed from data that includes the fraud/heavy IPs themselves. In production, compute baselines from a trusted historical window or use robust statistics.
- **The heavy-legit IPs are the false-positive test.** Check whether they show up in `flagged_spikes` or `flagged_ramps` — if they do, tune `Z_THRESHOLD` / `MIN_CONSECUTIVE_HOURS` / `GROWTH_THRESHOLD` against known-good heavy users before trusting the thresholds.
- **This is a heuristic, not a fraud model.** Real systems add subscriber identity, geolocation consistency, per-subscriber history, and usually a trained model layered on top of features like these.